In [1]:
import requests
import pandas as pd
from datetime import datetime
import time
from google.colab import files


API_KEY = "bJOlVzL7NTPYdAQX3da5ALMjNQ4WuMr9"  # 네 키
START_YEAR = 2014
END_YEAR = 2025

OUTPUT_CSV = "nyt_2014_2025_headlines.csv"

def fetch_month(year, month, api_key, max_retries=5):
    """
    특정 연/월 데이터를 받아온다.
    429(Too Many Requests) 발생 시 점점 더 길게 기다리면서 재시도.
    """
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    params = {"api-key": api_key}

    backoff = 10  # 첫 대기(초)

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
            status = resp.status_code
            print(f"[INFO] {year}-{month:02d} attempt {attempt} status: {status}")

            # 정상
            if status == 200:
                data = resp.json()
                docs = data.get("response", {}).get("docs", [])
                return docs

            # 레이트 리밋
            if status == 429:
                print(f"[WARN] {year}-{month:02d} rate limited (429). {backoff}초 대기 후 재시도.")
                time.sleep(backoff)
                backoff *= 2  # 다음엔 더 길게
                continue

            # 그 외 에러: 바로 실패 처리
            print(f"[WARN] {year}-{month:02d} 요청 실패 (status {status})")
            return []

        except Exception as e:
            print(f"[ERROR] {year}-{month:02d} 요청 중 예외: {e}")
            # 예외가 떠도 백오프 후 재시도
            time.sleep(backoff)
            backoff *= 2

    # 모든 시도 실패
    print(f"[FAIL] {year}-{month:02d} 최종 실패")
    return []


def extract_titles_from_docs(docs, year, month):
    """
    docs에서 우리가 쓰는 최소 필드만 추출
    """
    rows = []
    for d in docs:
        title = d.get("headline", {}).get("main", "")
        pub_date_raw = d.get("pub_date", "")

        if not title:
            continue

        # pub_date -> YYYY-MM-DD
        if pub_date_raw:
            try:
                dt = datetime.fromisoformat(pub_date_raw.replace("Z", "+00:00"))
                date_clean = dt.strftime("%Y-%m-%d")
            except Exception:
                date_clean = pub_date_raw[:10]
        else:
            date_clean = None

        rows.append({
            "year": year,
            "month": month,
            "date": date_clean,
            "headline": title.strip()
        })
    return rows


def main():
    all_rows = []

    for year in range(START_YEAR, END_YEAR + 1):
        for month in range(1, 13):
            docs = fetch_month(year, month, API_KEY)
            if not docs:
                # 실패했으면 그냥 비워두고 계속 진행
                continue

            month_rows = extract_titles_from_docs(docs, year, month)
            print(f"[INFO] {year}-{month:02d}: {len(month_rows)}건 추출됨")
            all_rows.extend(month_rows)

            # 월 사이 기본 대기 (API한테 숨 좀 쉬게)
            time.sleep(10)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df = (
            df
            .drop_duplicates(subset=["date", "headline"])
            .sort_values(["date", "headline"])
            .reset_index(drop=True)
        )

    print(f"\n[RESULT] 총 {len(df)}개 기사 헤드라인 수집 완료")
    print(df.head(10))

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[DONE] CSV 저장 완료 → {OUTPUT_CSV}")
    files.download(OUTPUT_CSV)

    return df

df_all = main()


[INFO] 2014-01 attempt 1 status: 200
[INFO] 2014-01: 7337건 추출됨
[INFO] 2014-02 attempt 1 status: 200
[INFO] 2014-02: 6941건 추출됨
[INFO] 2014-03 attempt 1 status: 200
[INFO] 2014-03: 7187건 추출됨
[INFO] 2014-04 attempt 1 status: 200
[INFO] 2014-04: 7344건 추출됨
[INFO] 2014-05 attempt 1 status: 200
[INFO] 2014-05: 7549건 추출됨
[INFO] 2014-06 attempt 1 status: 200
[INFO] 2014-06: 7345건 추출됨
[INFO] 2014-07 attempt 1 status: 429
[WARN] 2014-07 rate limited (429). 10초 대기 후 재시도.
[INFO] 2014-07 attempt 2 status: 200
[INFO] 2014-07: 6814건 추출됨
[INFO] 2014-08 attempt 1 status: 200
[INFO] 2014-08: 6577건 추출됨
[INFO] 2014-09 attempt 1 status: 200
[INFO] 2014-09: 7585건 추출됨
[INFO] 2014-10 attempt 1 status: 200
[INFO] 2014-10: 7861건 추출됨
[INFO] 2014-11 attempt 1 status: 200
[INFO] 2014-11: 6709건 추출됨
[INFO] 2014-12 attempt 1 status: 200
[INFO] 2014-12: 6701건 추출됨
[INFO] 2015-01 attempt 1 status: 200
[INFO] 2015-01: 6906건 추출됨
[INFO] 2015-02 attempt 1 status: 200
[INFO] 2015-02: 6437건 추출됨
[INFO] 2015-03 attempt 1 status:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>